In [ ]:
from __future__ import print_function
import datetime
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]

def get_calendar_events():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("client_secret_909543820885-ibsnmoblpu0r9ihagr85t1f10nlaq33s.apps.googleusercontent.com.json", SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    service = build("calendar", "v3", credentials=creds)

    now = datetime.datetime.utcnow().isoformat() + "Z" 
    events_result = service.events().list(
        calendarId="primary", timeMin=now,
        maxResults=5, singleEvents=True,
        orderBy="startTime").execute()
    events = events_result.get("items", [])

    return events


def summarize_events(events):
    if not events:
        return "No upcoming events found."

    events_text = ""
    for event in events:
        start = event["start"].get("dateTime", event["start"].get("date"))
        events_text += f"- {event['summary']} at {start}\n"

    prompt_template = PromptTemplate(
        input_variables=["content"],
        template="Summarize these calendar events as a reminder:\n\n{content}"
    )

    llm = OllamaLLM(model="llama3")  
    chain = LLMChain(llm=llm, prompt=prompt_template)
    summary = chain.run(content=events_text)

    return summary

if __name__ == "__main__":
    events = get_calendar_events()
    reminder = summarize_events(events)
    print("📅 Meeting Reminder:")
    print(reminder)


d:\PRACTISE\Crewai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=909543820885-ibsnmoblpu0r9ihagr85t1f10nlaq33s.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64346%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly&state=WIbgIOwmS07oEt8uDm66hGJpobJsZF&access_type=offline
